# `parcelsim` — Street Segment Demand (Boston)

Generates **street-segment-level** delivery demand from OSM building data.
No census data required — works for any city with OSM coverage.

Based on the AMZ_Project_Picasso methodology (Hörl & Benki, 2025):

```
service_time = 2 × parking_dist / walking_speed
             + indoors_time × (building_area / max_area)
             + handover_time
```

| Parameter | Default | Source |
|---|---|---|
| walking_speed | 1.4 m/s | AMZ calibration |
| handover_time | 60 s | AMZ calibration |
| indoors_time | 300 s | AMZ calibration |
| buffer_m | 30 m | buildings within 30m assigned to segment |

> **First run**: downloads OSM road network + buildings + parking. Cached to `parcelsim_cache/`.

In [ ]:
!pip install -q "parcelsim[viz]"

In [ ]:
import os
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

import parcelsim
os.makedirs('outputs', exist_ok=True)
print(f'parcelsim {parcelsim.__version__}')

CRS = 'EPSG:32619'  # UTM Zone 19N

## 1. City

In [ ]:
from parcelsim.city import City

city = City.from_osmnx(
    query='Boston, Massachusetts, USA',
    crs=CRS,
    country_iso='US',
)
print(city)

## 2. Segment demand — OSM buildings

In [ ]:
from parcelsim.demand.osm_segment_model import OSMSegmentDemandModel

model = OSMSegmentDemandModel(
    buffer_m=30.0,        # buildings within 30m assigned to segment
    walking_speed=1.4,    # m/s
    handover_time=60.0,   # seconds
    indoors_time=300.0,   # seconds
    cache_dir='parcelsim_cache',
)

segments = model.compute(city)
print(model.summary(segments))
print(f'\nSegment sample:')
print(segments[['segment_id','highway','segment_length','building_count','service_time','land_use']].head(10).to_string())

## 3. Map — service time per segment

In [ ]:
seg_3857 = segments[segments['service_time'] > 0].to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 10))
bounds = seg_3857.total_bounds
pad = 1_500
ax.set_xlim(bounds[0]-pad, bounds[2]+pad)
ax.set_ylim(bounds[1]-pad, bounds[3]+pad)
ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.CartoDB.Positron, zoom=13)
seg_3857.plot(
    column='service_time', ax=ax, cmap='YlOrRd',
    linewidth=1.2, legend=True,
    legend_kwds={'label': 'Service time (s)', 'shrink': 0.5}
)
ax.set_title('Boston — Delivery service time per street segment', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('outputs/boston_segments_01_service_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Map — land use per segment

In [ ]:
land_use_colors = {
    'residential':  '#3498db',
    'commercial':   '#e74c3c',
    'industrial':   '#e67e22',
    'university':   '#9b59b6',
    'mixed':        '#95a5a6',
    'unknown':      '#bdc3c7',
}

fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(bounds[0]-pad, bounds[2]+pad)
ax.set_ylim(bounds[1]-pad, bounds[3]+pad)
ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.CartoDB.Positron, zoom=13)

for lu, color in land_use_colors.items():
    subset = seg_3857[seg_3857['land_use'] == lu]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, linewidth=1.2, label=lu)

ax.legend(title='Land use', loc='lower right')
ax.set_title('Boston — Land use per street segment (from OSM buildings)', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('outputs/boston_segments_02_land_use.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Export

The segment GeoDataFrame is compatible with the AMZ_Project_Picasso `X.csv` format.

In [ ]:
# Save as CSV (AMZ-compatible format)
export_cols = ['segment_id','highway','is_oneway','segment_length',
               'building_count','land_use','parking_dist','service_time']
segments[export_cols].to_csv('outputs/boston_segments_X.csv', index=False)
print(f'Saved {len(segments):,} segments to outputs/boston_segments_X.csv')

# Save as GeoParquet for GIS use
segments.to_parquet('outputs/boston_segments.parquet')
print(f'Saved GeoParquet to outputs/boston_segments.parquet')